In [ ]:
import numpy as np
import pandas as pd

# Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T

from dataset import TrainDataset, TestDataset

image_size = 64
batch_size = 64
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_dataset = TrainDataset(root_path = './cs441-assn3-data/Train_64/', transform = train_transform)
test_dataset = TestDataset(root_path = './cs441-assn3-data/Test_64/', transform = eval_transform)

val_ratio = 0.2 # train 80%, val 20%

# 전체 길이 기준으로 train/val 길이 계산
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset_split, val_dataset = random_split(
    train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(dataset=train_dataset_split,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last = True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

# Your Awesome Model

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. ResNeXt34-SE Components
# ==========================================
class SEModule(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class ResNeXtBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None, cardinality=32, base_width=64, reduction=16):
        super().__init__()
        D = int(planes * (base_width / 64.0))
        C = cardinality
        self.conv1 = nn.Conv2d(inplanes, D * C, kernel_size=3, stride=stride, padding=1, groups=C, bias=False)
        self.bn1 = nn.BatchNorm2d(D * C)
        self.conv2 = nn.Conv2d(D * C, planes * self.expansion, kernel_size=3, stride=1, padding=1, groups=C, bias=False)
        self.bn2 = nn.BatchNorm2d(planes * self.expansion)
        self.se = SEModule(planes, reduction)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

class ResNeXt34(nn.Module):
    def __init__(self, num_classes=15, cardinality=8, base_width=64):
        super().__init__()
        self.inplanes = 64
        self.cardinality = cardinality
        self.base_width = base_width
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(64, 3, stride=1)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        layers = [ResNeXtBasicBlock(self.inplanes, planes, stride, downsample, self.cardinality, self.base_width)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(ResNeXtBasicBlock(self.inplanes, planes, stride=1, cardinality=self.cardinality, base_width=self.base_width))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ==========================================
# 2. ConvNeXt-Small Components
# ==========================================
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        self.normalized_shape = (normalized_shape, )
    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x

class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0. or not self.training: return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

class Block(nn.Module):
    def __init__(self, dim, drop_path=0.):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim) 
        self.norm = LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True) if dim > 0 else None
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None: x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        x = input + self.drop_path(x)
        return x

class ConvNeXt(nn.Module):
    def __init__(self, in_chans=3, num_classes=15, depths=[3, 3, 27, 3], dims=[96, 192, 384, 768], drop_path_rate=0.4):
        super().__init__()
        self.downsample_layers = nn.ModuleList() 
        stem = nn.Sequential(nn.Conv2d(in_chans, dims[0], kernel_size=3, stride=1, padding=1), LayerNorm(dims[0], eps=1e-6, data_format="channels_first"))
        self.downsample_layers.append(stem)
        for i in range(3):
            downsample_layer = nn.Sequential(LayerNorm(dims[i], eps=1e-6, data_format="channels_first"), nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2))
            self.downsample_layers.append(downsample_layer)
        self.stages = nn.ModuleList() 
        dp_rates = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))] 
        cur = 0
        for i in range(4):
            stage = nn.Sequential(*[Block(dim=dims[i], drop_path=dp_rates[cur + j]) for j in range(depths[i])])
            self.stages.append(stage)
            cur += depths[i]
        self.norm = LayerNorm(dims[-1], eps=1e-6, data_format="channels_first")
        self.head = nn.Linear(dims[-1], num_classes)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x):
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        x = self.norm(x.mean([-2, -1], keepdim=True))
        x = x.flatten(1)
        x = self.head(x)
        return x

# Ensemble Model
class EnsembleModel(nn.Module):
    def __init__(self, num_classes=15):
        super().__init__()
        self.resnext = ResNeXt34(num_classes=num_classes, cardinality=8, base_width=64)
        
        # ConvNeXt-Small (~50M Params)
        self.convnext = ConvNeXt(num_classes=num_classes, drop_path_rate=0.4)
        
    def forward(self, x):
        out1 = self.resnext(x)
        out2 = self.convnext(x)
        # 평균 앙상블
        return (out1 + out2) / 2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EnsembleModel(num_classes=15).to(device)

# Model parameter checking

In [ ]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

# Model training

In [ ]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

In [ ]:
import tqdm
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR
from torch.cuda.amp import autocast

# 설정
epochs = 15
save_path = "best_ensemble_model.pth"
best_val_loss = float("inf")

# Optimizer & Scheduler
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.05)
scaler = torch.amp.GradScaler('cuda')

# Scheduler: Warmup (3.5 epochs) + Cosine
total_steps = epochs * len(train_loader)
warmup_steps = 1000
scheduler = SequentialLR(
    optimizer, 
    schedulers=[
        LambdaLR(optimizer, lambda step: (step + 1) / warmup_steps),
        CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps)
    ], 
    milestones=[warmup_steps]
)

print("Start Training Ensemble Model...")

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss, train_acc = 0.0, 0.0
    total = 0
    
    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            output = model(x)
            loss = criterion(output, y)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step() # Step per batch
        
        train_loss += loss.item() * y.size(0)
        train_acc += (output.argmax(1) == y).sum().item()
        total += y.size(0)
        
    print(f"Epoch {epoch} | Train Loss: {train_loss/total:.4f} | Acc: {train_acc/total:.4f}")

    # VALIDATION
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    total = 0
    
    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x, y = x.to(device), y.to(device)
            with autocast():
                output = model(x)
                loss = criterion(output, y)
            val_loss += loss.item() * y.size(0)
            val_acc += (output.argmax(1) == y).sum().item()
            total += y.size(0)
            
    val_loss /= total
    val_acc /= total
    print(f"Epoch {epoch} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"Saved Best Model (Val Loss: {val_loss:.4f})")

In [ ]:
'''
# 모델 불러오기
checkpoint = torch.load("best_model.pth", map_location=device)

# 먼저 순수 모델을 만들고 로드
base_model = ConvNeXtBN(num_classes=15)
base_model.load_state_dict(checkpoint["model_state_dict"])

# 그 다음에 DataParallel로 감쌈
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
'''

In [ ]:
'''
# 모델 로드 검증
missing_keys, unexpected_keys = base_model.load_state_dict(
    checkpoint["model_state_dict"], strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

# 파라미터 값 확인
with torch.no_grad():
    w = base_model.downsample_layers[0][0].weight

print("Sample weight stats:")
print("  mean:", w.mean().item())
print("  std :", w.std().item())
print("  min :", w.min().item())
print("  max :", w.max().item())

# forward 테스트
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out = model(dummy)

print("Output shape:", out.shape)
print("Has NaN:", torch.isnan(out).any().item())
'''

In [ ]:
'''
print("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))
print("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))
print("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))
'''

# Submit
Do not edit the submission code below.

In [ ]:
import pandas as pd

# Load Best Model
model.load_state_dict(torch.load(save_path))
model.eval()

submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')
predictions = []

print("Generating submission with TTA...")

with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = x.to(device)
        
        # 1. Original Prediction
        out1 = model(x)
        prob1 = F.softmax(out1, dim=1)
        
        # 2. Horizontal Flip Prediction (TTA)
        x_flip = torch.flip(x, dims=[3])
        out2 = model(x_flip)
        prob2 = F.softmax(out2, dim=1)
        
        # 3. Ensemble (Average)
        final_prob = (prob1 + prob2) / 2
        pred = final_prob.argmax(dim=1)
        predictions.extend(pred.cpu().numpy())

submit['label'] = predictions
submit.to_csv('submission_ensemble_tta.csv', index=False)
print("Done! 'submission_ensemble_tta.csv' created.")